# NUS ST4234 Bayesian Statistics — Modern Python Case Study
## Titanic survival with PyMC, ArviZ, Bambi and Bokeh

This notebook is designed as a **teaching case study**, not merely a collection of library calls.

We start with a question that can be solved analytically, so the meaning of a posterior is transparent. We then deliberately increase model complexity until exact integration becomes inconvenient and modern Bayesian computation becomes useful.

### Case-study question

> **How should uncertainty about Titanic survival probabilities be represented, updated, pooled across related passenger groups, and propagated into predictions?**

The notebook uses the Titanic passenger dataset commonly used in the Kaggle competition. The online source mirrors the Kaggle training schema. An embedded compressed fallback is included so the notebook can still run when the URL is unavailable.

### Bayesian software stack

| Package | Role in this notebook | Why it is appropriate |
|---|---|---|
| **PyMC** | probabilistic models, NUTS/MCMC, posterior predictive sampling, ADVI | general-purpose Bayesian probabilistic programming |
| **ArviZ** | posterior summaries, R-hat, ESS, MCSE, LOO | Bayesian diagnostics and model evaluation |
| **Bambi** | formula-based Bayesian logistic regression | concise GLM/GLMM layer on top of PyMC |
| **SciPy** | exact conjugate distributions and special functions | ideal when a closed form exists |
| **Bokeh** | interactive teaching visualizations | inspect posterior distributions and uncertainty interactively |
| **pandas** | tables and feature preparation | interpretable tabular workflow |

The implementation targets the **2026-era stack**: PyMC 6.x, Bambi 0.19+, ArviZ 1.2+ and Bokeh 3.8+.

## Learning objectives and ST4234 map

By the end of the notebook you should be able to connect each computational operation to a statistical idea.

| ST4234 concept | Titanic question used to teach it | Main tool |
|---|---|---|
| Prior, likelihood, posterior | What is the overall survival probability? | SciPy / PyMC |
| Conjugacy | How does a Beta prior update after Binomial data? | SciPy |
| Credible intervals | What survival probabilities remain plausible? | SciPy / ArviZ |
| Prior sensitivity | When does prior information materially influence inference? | SciPy + Bokeh |
| Posterior predictive distribution | How many of the next 100 comparable passengers might survive? | SciPy / PyMC |
| Bayesian testing / Bayes factor | Does the evidence favor `p != 0.5` over `p = 0.5`? | SciPy |
| Normal Bayesian models | What is the mean age and its uncertainty? | PyMC |
| MCMC | How do we sample a posterior that is not analytically convenient? | PyMC |
| Metropolis-Hastings | What changes when a generic random-walk sampler is used? | PyMC `Metropolis` |
| NUTS / HMC | How does a modern gradient-based sampler explore continuous posteriors? | PyMC |
| MCMC diagnostics | Did the chains actually explore the target distribution? | ArviZ |
| Multivariate normal | How do age and log-fare vary jointly? | PyMC `MvNormal` + LKJ |
| Hierarchical Bayes | How should Sex × Pclass survival rates share information? | PyMC |
| Partial pooling | Why should small groups shrink more than large groups? | PyMC + Bokeh |
| Bayesian regression / GLM | How does survival probability depend on passenger features? | Bambi |
| Variational Bayes | Can posterior approximation be turned into optimization? | PyMC ADVI |
| Model comparison | Is an interaction model predictively better? | ArviZ PSIS-LOO |
| Posterior predictive checking | Can fitted models reproduce patterns in the observed data? | Bambi/PyMC + Bokeh |

## 0. Environment and reproducibility

Bayesian software evolves more quickly than the mathematics. For reproducibility, keep the packages mutually compatible.

If your environment is missing the Bayesian packages, **uncomment and run the installation line once**, restart the kernel, then continue.

Bambi 0.18 moved to PyMC 6, and Bambi 0.19 is the current 2026 release line used here. PyMC's current `pm.sample()` assigns NUTS to suitable continuous variables and accepts NUTS settings through `nuts={...}`.

In [ ]:
# Run once if needed, then restart the kernel.
# %pip install -U "pymc>=6.3,<7" "bambi>=0.19,<1" "arviz>=1.2,<2" \
#     "bokeh>=3.8,<4" "pandas>=2.2" "scipy>=1.11" "scikit-learn>=1.4"

In [ ]:
from __future__ import annotations

import base64
import gzip
import io
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.special import betaln, expit
from scipy.stats import beta as beta_dist
from scipy.stats import betabinom

import pymc as pm
import arviz as az
import bambi as bmb

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import ColumnDataSource, HoverTool, Span
from bokeh.palettes import Category10
from bokeh.plotting import figure

from sklearn.metrics import brier_score_loss, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
output_notebook()

SEED = 4234
RNG = np.random.default_rng(SEED)

print("PyMC :", pm.__version__)
print("ArviZ:", az.__version__)
print("Bambi:", bmb.__version__)

### Sampling configuration: exploration versus final inference

MCMC accuracy depends on **effective posterior draws**, not merely raw iteration count. During learning, shorter runs are convenient. For final reporting, use more chains and draws.

- `FAST_MODE=True`: suitable for walking through the notebook.
- `FAST_MODE=False`: stronger default for a final analysis.

Changing this flag changes computation, **not the statistical model**.

In [ ]:
@dataclass(frozen=True)
class SamplingConfig:
    fast_mode: bool = True

    @property
    def chains(self) -> int:
        return 2 if self.fast_mode else 4

    @property
    def draws(self) -> int:
        return 500 if self.fast_mode else 1500

    @property
    def tune(self) -> int:
        return 500 if self.fast_mode else 1500

    @property
    def advi_steps(self) -> int:
        return 8_000 if self.fast_mode else 30_000

    @property
    def target_accept(self) -> float:
        return 0.90 if self.fast_mode else 0.95

CFG = SamplingConfig(fast_mode=True)
CFG

## 1. Load the Titanic data

The raw training table has one row per passenger and a binary target:

\[
Y_i = \begin{cases}
1 & \text{passenger } i \text{ survived}\\
0 & \text{otherwise.}
\end{cases}
\]

That makes Bernoulli/Binomial likelihoods the natural starting point.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
EMBEDDED_TITANIC_GZ_B64 = '...'


def load_titanic() -> tuple[pd.DataFrame, str]:
    try:
        df = pd.read_csv(DATA_URL)
        return df, "online mirror of Kaggle Titanic training data"
    except Exception:
        raw = gzip.decompress(base64.b64decode(EMBEDDED_TITANIC_GZ_B64))
        return pd.read_csv(io.BytesIO(raw)), "embedded offline fallback"


df, data_source = load_titanic()
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
data_dictionary = pd.DataFrame({
    "variable": ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"],
    "Bayesian role": [
        "Bernoulli outcome", "categorical predictor / grouping variable", "categorical predictor / grouping variable",
        "continuous predictor", "family-size component", "family-size component", "continuous predictor", "categorical predictor"
    ],
    "interpretation": [
        "1=survived, 0=died", "1/2/3 passenger class", "recorded passenger sex", "age in years",
        "siblings/spouses aboard", "parents/children aboard", "ticket fare", "embarkation port"
    ]
})
display(data_dictionary)

missing = df.isna().sum().rename("missing").to_frame()
missing["pct"] = 100 * missing["missing"] / len(df)
display(missing.sort_values("missing", ascending=False).head(10).round(2))

### Exploratory table: why a one-parameter model will be incomplete

If all passengers were exchangeable, one common survival probability \(p\) might be adequate. But class and sex visibly define very different subpopulations. We begin with a simple model **on purpose**, learn its mechanics, and then relax its assumptions.

In [ ]:
overall = pd.DataFrame({
    "passengers": [len(df)],
    "survivors": [int(df["Survived"].sum())],
    "observed_rate": [df["Survived"].mean()]
})

group_rates = (
    df.groupby(["Sex", "Pclass"], observed=True)["Survived"]
      .agg(passengers="size", survivors="sum", observed_rate="mean")
      .reset_index()
)

display(overall.round(4))
display(group_rates.round(4))

In [ ]:
plot_df = group_rates.copy()
plot_df["group"] = plot_df["Sex"].astype(str) + " / class " + plot_df["Pclass"].astype(str)
p = figure(width=760, height=380, title="Observed Titanic survival rates by Sex × Pclass",
           x_range=plot_df["group"].tolist(),
           x_axis_label="group", y_axis_label="observed survival rate", y_range=(0, 1))
p.vbar(x=plot_df["group"], top=plot_df["observed_rate"], width=0.7)
p.add_tools(HoverTool(tooltips=[("group", "@x"), ("rate", "@top{0.000}")]))
p.xaxis.major_label_orientation = 0.8
show(p)

# Part I — Exact Bayesian inference
## 2. Beta–Binomial conjugacy

Let \(p\) denote the survival probability for a randomly selected passenger from this population.

\[
Y_i\mid p\sim\operatorname{Bernoulli}(p),
\qquad
S=\sum_iY_i\mid p\sim\operatorname{Binomial}(n,p).
\]

Choose a Beta prior:

\[
p\sim\operatorname{Beta}(\alpha_0,\beta_0).
\]

Because the Beta distribution is conjugate to the Binomial likelihood,

\[
p\mid S=s
\sim
\operatorname{Beta}(\alpha_0+s,\,\beta_0+n-s).
\]

No MCMC is required here. A specialized Bayesian package should not replace algebra when the algebra is exact and simpler.

In [ ]:
n = len(df)
s = int(df["Survived"].sum())

alpha0, beta0 = 2.0, 2.0
alpha_post = alpha0 + s
beta_post = beta0 + n - s

post_mean = alpha_post / (alpha_post + beta_post)
post_map = (alpha_post - 1) / (alpha_post + beta_post - 2)
ci95 = beta_dist.ppf([0.025, 0.975], alpha_post, beta_post)
prob_gt_04 = 1 - beta_dist.cdf(0.40, alpha_post, beta_post)

posterior_table = pd.DataFrame({
    "quantity": ["observed rate", "prior mean", "posterior mean", "posterior MAP", "95% credible lower", "95% credible upper", "P(p > 0.40 | data)"],
    "value": [s/n, alpha0/(alpha0+beta0), post_mean, post_map, ci95[0], ci95[1], prob_gt_04],
    "interpretation": [
        "sample proportion", "belief before current Titanic data", "Bayesian squared-error estimate",
        "posterior mode", "2.5% posterior quantile", "97.5% posterior quantile",
        "direct posterior probability of a scientifically meaningful statement"
    ]
})
display(posterior_table.round(4))

In [ ]:
grid = np.linspace(0.15, 0.60, 700)
prior_pdf = beta_dist.pdf(grid, alpha0, beta0)
post_pdf = beta_dist.pdf(grid, alpha_post, beta_post)

p = figure(width=780, height=400, title="Prior → posterior update for overall survival probability",
           x_axis_label="survival probability p", y_axis_label="density")
p.line(grid, prior_pdf, line_width=2, legend_label="Beta(2,2) prior")
p.line(grid, post_pdf, line_width=3, legend_label="posterior")
p.add_layout(Span(location=s/n, dimension="height", line_dash="dashed", line_width=2))
p.add_tools(HoverTool(tooltips=[("p", "$x{0.000}"), ("density", "$y{0.000}")]))
p.legend.location = "top_left"
show(p)

### Interpretation

The posterior distribution is the important object—not merely its mean.

A Bayesian credible interval can be read directly as a probability statement about \(p\): conditional on the assumed model, prior and observed data, 95% of posterior mass lies inside the reported interval.

The posterior is also much narrower than the prior because 891 observations contain far more information than the weak \(\operatorname{Beta}(2,2)\) prior.

## 3. Prior sensitivity

Priors matter most when the likelihood is weak. We demonstrate this twice:

1. use all Titanic passengers;
2. use only the first 12 passengers as an artificial small-data scenario.

This is more informative than arguing abstractly about whether priors are "subjective".

In [ ]:
priors = {
    "Uniform Beta(1,1)": (1, 1),
    "Weak centered Beta(2,2)": (2, 2),
    "Optimistic Beta(8,2)": (8, 2),
    "Pessimistic Beta(2,8)": (2, 8),
}

small = df.iloc[:12]
scenarios = {
    "small n=12": (len(small), int(small["Survived"].sum())),
    "full n=891": (n, s),
}

rows = []
for scenario, (nn, ss) in scenarios.items():
    for label, (a, b) in priors.items():
        aa, bb = a + ss, b + nn - ss
        lo, hi = beta_dist.ppf([0.025, 0.975], aa, bb)
        rows.append({
            "scenario": scenario, "prior": label,
            "prior_mean": a/(a+b), "posterior_mean": aa/(aa+bb),
            "lower": lo, "upper": hi
        })

sensitivity = pd.DataFrame(rows)
display(sensitivity.round(4))

In [ ]:
plots = []
for scenario in scenarios:
    sub = sensitivity[sensitivity["scenario"] == scenario].copy()
    p = figure(width=760, height=320, title=f"Prior sensitivity — {scenario}", x_range=(0, 1),
               y_range=list(reversed(sub["prior"].tolist())), x_axis_label="posterior survival probability")
    y = sub["prior"].tolist()
    p.segment(x0=sub["lower"], x1=sub["upper"], y0=y, y1=y, line_width=4)
    p.scatter(x=sub["posterior_mean"], y=y, size=10)
    plots.append(p)
show(column(*plots))

### What this experiment teaches

For the small sample, the same data can lead to noticeably different posterior beliefs under different priors. With the full Titanic dataset, the posterior means become much closer because the likelihood dominates.

This is the general principle:

\[
\text{weak data} \Rightarrow \text{prior has leverage},
\qquad
\text{strong data} \Rightarrow \text{likelihood dominates}.
\]

## 4. Posterior predictive distribution

Bayesian prediction integrates parameter uncertainty rather than plugging in a single \(\hat p\).

For \(m\) future exchangeable passengers,

\[
\tilde S\mid p\sim\operatorname{Binomial}(m,p),
\qquad
p\mid y\sim\operatorname{Beta}(\alpha_n,\beta_n).
\]

After integrating over \(p\),

\[
\tilde S\mid y\sim\operatorname{BetaBinomial}(m,\alpha_n,\beta_n).
\]

In [ ]:
m_future = 100
k = np.arange(m_future + 1)
pred_pmf = betabinom.pmf(k, m_future, alpha_post, beta_post)
q025, q50, q975 = betabinom.ppf([0.025, 0.50, 0.975], m_future, alpha_post, beta_post)

pred_summary = pd.DataFrame({
    "quantity": ["predictive mean survivors", "predictive median", "95% predictive lower", "95% predictive upper"],
    "value": [m_future * post_mean, q50, q025, q975]
})
display(pred_summary.round(2))

p = figure(width=800, height=380, title="Posterior predictive distribution: survivors among next 100 comparable passengers",
           x_axis_label="number surviving", y_axis_label="posterior predictive probability")
p.vbar(x=k, top=pred_pmf, width=0.9)
p.add_layout(Span(location=q025, dimension="height", line_dash="dashed"))
p.add_layout(Span(location=q975, dimension="height", line_dash="dashed"))
show(p)

## 5. Bayesian hypothesis testing with an exact Bayes factor

Consider:

\[
H_0:p=0.5
\]

versus

\[
H_1:p\sim\operatorname{Beta}(2,2).
\]

The Bayes factor is

\[
BF_{10}=\frac{p(y\mid H_1)}{p(y\mid H_0)}.
\]

For the Beta-Binomial alternative the marginal likelihood is analytic, so again no sampler is needed.

**Important:** a Bayes factor compares complete probability models. Changing the prior under \(H_1\) changes the Bayes factor.

In [ ]:
# The Binomial coefficient is common to H0 and H1 and cancels in BF10.
log_marginal_h1 = betaln(alpha0 + s, beta0 + n - s) - betaln(alpha0, beta0)
log_marginal_h0 = s * np.log(0.5) + (n - s) * np.log(0.5)
log_bf10 = log_marginal_h1 - log_marginal_h0
bf10 = np.exp(log_bf10)

bf_table = pd.DataFrame({
    "quantity": ["log BF10", "BF10"],
    "value": [log_bf10, bf10],
    "interpretation": ["log evidence ratio H1/H0", "evidence multiplier H1 relative to H0"]
})
display(bf_table)

# Part II — PyMC and modern MCMC
## 6. Re-express the simple Beta–Binomial model in PyMC

Why fit a model with MCMC when we already know its exact posterior?

Because it gives us a **ground truth validation exercise**. A reliable sampler should reproduce the known Beta posterior.

This is a useful habit in computational statistics: test complicated machinery on a problem whose answer is already known.

In [ ]:
with pm.Model() as beta_binomial_model:
    p_survive = pm.Beta("p_survive", alpha=alpha0, beta=beta0)
    y_survive = pm.Binomial("y_survive", n=n, p=p_survive, observed=s)

    idata_nuts_simple = pm.sample(
        draws=CFG.draws,
        tune=CFG.tune,
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        random_seed=SEED,
        nuts={"target_accept": CFG.target_accept},
        progressbar=True,
    )

summary_simple = az.summary(idata_nuts_simple, var_names=["p_survive"], round_to=4)
display(summary_simple)
print("Exact posterior mean:", round(post_mean, 6))

In [ ]:
nuts_samples = np.asarray(idata_nuts_simple.posterior["p_survive"]).reshape(-1)

p = figure(width=780, height=380, title="PyMC NUTS posterior versus exact Beta posterior",
           x_axis_label="p", y_axis_label="density")
hist, edges = np.histogram(nuts_samples, bins=45, density=True)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.35, legend_label="NUTS samples")
p.line(grid, post_pdf, line_width=3, legend_label="exact Beta posterior")
p.legend.location = "top_left"
show(p)

## 7. Metropolis-Hastings using PyMC—not a hand-written sampler

Metropolis-Hastings is foundational in ST4234. We can study it without implementing the transition kernel ourselves.

PyMC's generic `Metropolis` step method uses a random-walk proposal. Compared with NUTS, adjacent samples are usually more autocorrelated for smooth continuous targets.

This section therefore compares **algorithms while holding the statistical model fixed**.

In [ ]:
with beta_binomial_model:
    metropolis_step = pm.Metropolis()
    idata_mh_simple = pm.sample(
        draws=CFG.draws * 2,
        tune=CFG.tune,
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        step=metropolis_step,
        random_seed=SEED,
        progressbar=True,
    )

comparison_diag = pd.concat([
    az.summary(idata_nuts_simple, var_names=["p_survive"], round_to=4).assign(sampler="NUTS"),
    az.summary(idata_mh_simple, var_names=["p_survive"], round_to=4).assign(sampler="Metropolis")
]).reset_index(names="parameter")

display(comparison_diag[[c for c in comparison_diag.columns if c in
                         ["parameter","sampler","mean","sd","ess_bulk","ess_tail","r_hat"]]])

### Gibbs sampling: where it fits

Gibbs sampling repeatedly draws each block from its **full conditional distribution**:

\[
\theta_1^{(t+1)}\sim p(\theta_1\mid\theta_2^{(t)},y),
\qquad
\theta_2^{(t+1)}\sim p(\theta_2\mid\theta_1^{(t+1)},y).
\]

It can be interpreted as a special Metropolis-Hastings construction whose proposed full-conditional draws are accepted with probability 1.

Modern PyMC automatically assigns specialized steps according to variable type; for many differentiable continuous posteriors, **NUTS is preferable to manually assembling a Gibbs sampler**. Gibbs remains extremely important conceptually and is still valuable for models with convenient conditional distributions or discrete latent variables.

## 8. MCMC diagnostics: never trust a trace only because sampling finished

For continuous MCMC models we care about at least four ideas:

| Diagnostic | Question |
|---|---|
| **R-hat** | Did independent chains mix to the same stationary distribution? |
| **ESS** | How many independent-equivalent draws does the autocorrelated chain contain? |
| **MCSE** | How much numerical error remains in a posterior estimate because sampling is finite? |
| **divergences** | Did Hamiltonian trajectories encounter problematic posterior geometry? |

A useful practical target is R-hat very close to 1, healthy ESS relative to chain length, and zero divergences. These are diagnostics—not proofs of model correctness.

In [ ]:
def diagnostic_table(idata, var_names=None):
    out = az.summary(idata, var_names=var_names, round_to=4)
    cols = [c for c in ["mean", "sd", "mcse_mean", "mcse_sd", "ess_bulk", "ess_tail", "r_hat"] if c in out.columns]
    return out[cols]


def divergence_count(idata) -> int:
    if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:
        return int(np.asarray(idata.sample_stats["diverging"]).sum())
    return 0


display(diagnostic_table(idata_nuts_simple, ["p_survive"]))
print("NUTS divergences:", divergence_count(idata_nuts_simple))

In [ ]:
chains = np.asarray(idata_nuts_simple.posterior["p_survive"])
p_trace = figure(width=850, height=340, title="Trace plot — each chain should explore the same region",
                 x_axis_label="draw", y_axis_label="p_survive")
for c in range(chains.shape[0]):
    p_trace.line(np.arange(chains.shape[1]), chains[c], alpha=0.8, legend_label=f"chain {c}")
p_trace.legend.click_policy = "hide"
show(p_trace)

## 9. A Normal Bayesian model: passenger age

ST4234 also studies Normal models. For passengers with observed age, consider

\[
Age_i\sim N(\mu,\sigma^2).
\]

We use weakly informative priors

\[
\mu\sim N(30,15^2),
\qquad
\sigma\sim\operatorname{HalfNormal}(15).
\]

This is intentionally simple. The Normal likelihood will not perfectly represent the real age distribution; later posterior predictive checks teach us to inspect that mismatch rather than pretending every fitted distribution is correct.

In [ ]:
age = df["Age"].dropna().to_numpy(dtype=float)

with pm.Model() as age_model:
    mu_age = pm.Normal("mu_age", mu=30, sigma=15)
    sigma_age = pm.HalfNormal("sigma_age", sigma=15)
    age_obs = pm.Normal("age_obs", mu=mu_age, sigma=sigma_age, observed=age)

    age_prior = pm.sample_prior_predictive(draws=500, random_seed=SEED)
    idata_age = pm.sample(
        draws=CFG.draws,
        tune=CFG.tune,
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        random_seed=SEED,
        nuts={"target_accept": CFG.target_accept},
    )
    pm.sample_posterior_predictive(idata_age, extend_inferencedata=True, random_seed=SEED)


display(diagnostic_table(idata_age, ["mu_age", "sigma_age"]))
print("Divergences:", divergence_count(idata_age))

In [ ]:
mu_draws = np.asarray(idata_age.posterior["mu_age"]).reshape(-1)
sigma_draws = np.asarray(idata_age.posterior["sigma_age"]).reshape(-1)

figs = []
for name, draws in [("mu_age", mu_draws), ("sigma_age", sigma_draws)]:
    hist, edges = np.histogram(draws, bins=40, density=True)
    p = figure(width=390, height=320, title=f"Posterior: {name}", x_axis_label=name, y_axis_label="density")
    p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.5)
    figs.append(p)
show(gridplot([figs]))

# Part III — Multivariate Bayesian modelling
## 10. A bivariate Normal model for Age and log(Fare)

A univariate Normal model describes one variable at a time. A multivariate Normal model additionally represents **dependence**.

Let

\[
x_i = \begin{bmatrix}z(Age_i)\\z(\log(1+Fare_i))\end{bmatrix}.
\]

We model

\[
x_i\sim MVN(\mu,\Sigma).
\]

Instead of placing a difficult prior directly on \(\Sigma\), modern Bayesian practice often separates standard deviations from correlations. PyMC's `LKJCholeskyCov` provides a prior over a Cholesky factor whose implied correlation matrix follows an LKJ prior.

This is an advanced but useful connection between ST4234 multivariate Normal theory and contemporary probabilistic programming.

In [ ]:
mv = df[["Age", "Fare"]].dropna().copy()
mv["logFare"] = np.log1p(mv["Fare"])
X_mv = mv[["Age", "logFare"]].to_numpy(float)
X_mv = (X_mv - X_mv.mean(axis=0)) / X_mv.std(axis=0, ddof=1)

coords_mv = {"axis": ["z_age", "z_logfare"], "obs_id": np.arange(len(X_mv))}

with pm.Model(coords=coords_mv) as mvn_model:
    mu = pm.Normal("mu", 0, 1.5, dims="axis")
    sd_dist = pm.Exponential.dist(1.0, shape=2)
    chol, corr, stds = pm.LKJCholeskyCov(
        "chol", n=2, eta=2.0, sd_dist=sd_dist, compute_corr=True
    )
    obs = pm.MvNormal("obs", mu=mu, chol=chol, observed=X_mv, dims=("obs_id", "axis"))

    idata_mvn = pm.sample(
        draws=max(300, CFG.draws // 2),
        tune=max(300, CFG.tune // 2),
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        random_seed=SEED,
        nuts={"target_accept": max(0.92, CFG.target_accept)},
    )


display(az.summary(idata_mvn, var_names=["mu", "chol_corr", "chol_stds"], round_to=3))
print("Divergences:", divergence_count(idata_mvn))

In [ ]:
corr_draws = np.asarray(idata_mvn.posterior["chol_corr"])[..., 0, 1].reshape(-1)
hist, edges = np.histogram(corr_draws, bins=40, density=True)
p = figure(width=760, height=350, title="Posterior correlation: standardized Age vs log(Fare)",
           x_axis_label="correlation", y_axis_label="density", x_range=(-1, 1))
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.5)
p.add_layout(Span(location=0, dimension="height", line_dash="dashed"))
show(p)

# Part IV — Hierarchical Bayes and partial pooling
## 11. Why hierarchical modelling is needed

The six `Sex × Pclass` groups have different sample sizes. A raw proportion treats each group independently:

\[
\hat p_g=\frac{s_g}{n_g}.
\]

That is **no pooling**.

At the other extreme, a single common \(p\) for everybody is **complete pooling**.

A hierarchical model occupies the useful middle ground:

\[
s_g\mid\theta_g\sim Binomial(n_g,\theta_g),
\]

\[
\theta_g\sim Beta(\mu\kappa,(1-\mu)\kappa),
\]

with hyperpriors

\[
\mu\sim Beta(2,2),
\qquad
\kappa\sim Gamma(2,0.1).
\]

Here:

- \(\mu\) is the population-level center;
- \(\kappa\) controls how tightly group probabilities cluster around that center;
- each \(\theta_g\) is allowed to differ, but groups learn from one another.

In [ ]:
hier = group_rates.copy()
hier["group"] = hier["Sex"].astype(str) + " / class " + hier["Pclass"].astype(str)
coords_h = {"group": hier["group"].tolist()}

with pm.Model(coords=coords_h) as hierarchical_model:
    mu_pop = pm.Beta("mu_pop", alpha=2, beta=2)
    kappa = pm.Gamma("kappa", alpha=2, beta=0.1)

    alpha_group = pm.Deterministic("alpha_group", mu_pop * kappa)
    beta_group = pm.Deterministic("beta_group", (1 - mu_pop) * kappa)

    theta = pm.Beta("theta", alpha=alpha_group, beta=beta_group, dims="group")
    survived = pm.Binomial(
        "survived", n=hier["passengers"].to_numpy(), p=theta,
        observed=hier["survivors"].to_numpy(), dims="group"
    )

    prior_h = pm.sample_prior_predictive(draws=500, random_seed=SEED)
    idata_h = pm.sample(
        draws=CFG.draws,
        tune=CFG.tune,
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        random_seed=SEED,
        nuts={"target_accept": max(0.93, CFG.target_accept)},
    )


display(diagnostic_table(idata_h, ["mu_pop", "kappa"]))
print("Divergences:", divergence_count(idata_h))

In [ ]:
theta_draws = np.asarray(idata_h.posterior["theta"])  # chain, draw, group
theta_flat = theta_draws.reshape(-1, theta_draws.shape[-1])

hier_out = hier.copy()
hier_out["posterior_mean"] = theta_flat.mean(axis=0)
hier_out["lower"] = np.quantile(theta_flat, 0.025, axis=0)
hier_out["upper"] = np.quantile(theta_flat, 0.975, axis=0)
hier_out["shrinkage"] = hier_out["posterior_mean"] - hier_out["observed_rate"]

pop_mean = float(np.asarray(idata_h.posterior["mu_pop"]).mean())
hier_out["distance_raw_to_pop"] = (hier_out["observed_rate"] - pop_mean).abs()
hier_out["distance_post_to_pop"] = (hier_out["posterior_mean"] - pop_mean).abs()

display(hier_out[["group", "passengers", "survivors", "observed_rate", "posterior_mean", "lower", "upper", "shrinkage"]].round(4))

In [ ]:
p = figure(width=850, height=420, title="Partial pooling: raw group rate → hierarchical posterior mean",
           x_axis_label="raw observed rate", y_axis_label="hierarchical posterior mean",
           x_range=(0,1), y_range=(0,1))
p.line([0,1], [0,1], line_dash="dashed", line_width=2, legend_label="no shrinkage")
source = ColumnDataSource(hier_out)
p.scatter(x="observed_rate", y="posterior_mean", size=12, source=source)
p.segment(x0="observed_rate", y0="posterior_mean", x1="posterior_mean", y1="posterior_mean", source=source, alpha=0.5)
p.add_tools(HoverTool(tooltips=[
    ("group", "@group"), ("n", "@passengers"),
    ("raw", "@observed_rate{0.000}"), ("posterior", "@posterior_mean{0.000}"),
    ("95% interval", "[@lower{0.000}, @upper{0.000}]")
]))
p.legend.location = "top_left"
show(p)

### Reading shrinkage correctly

Partial pooling does **not** force every group to be similar. It asks how much the data justify treating a group's apparent extremeness as real.

A small group usually has a broad likelihood. The shared population distribution therefore has more influence and the group estimate shrinks more.

A large group has a concentrated likelihood. Its own data dominate and little shrinkage is required.

This is one Bayesian route to regularization:

\[
\text{less information} \Rightarrow \text{stronger pooling},
\qquad
\text{more information} \Rightarrow \text{more autonomy}.
\]

## 12. Hierarchical posterior predictive checks

A posterior predictive check asks:

> If the fitted model were the data-generating mechanism, what replicated datasets would it produce?

For each posterior draw we simulate replicated survivor counts for every group and compare them with the actual counts/rates.

A model may have converged perfectly computationally and still be scientifically inadequate. **Convergence and model adequacy are different questions.**

In [ ]:
with hierarchical_model:
    pm.sample_posterior_predictive(idata_h, var_names=["survived"], extend_inferencedata=True, random_seed=SEED)

pp = np.asarray(idata_h.posterior_predictive["survived"])  # chain, draw, group
pp_flat = pp.reshape(-1, pp.shape[-1])
pp_rates = pp_flat / hier["passengers"].to_numpy()[None, :]

ppc_h = hier[["group", "passengers", "observed_rate"]].copy()
ppc_h["pred_mean"] = pp_rates.mean(axis=0)
ppc_h["pred_lower"] = np.quantile(pp_rates, 0.025, axis=0)
ppc_h["pred_upper"] = np.quantile(pp_rates, 0.975, axis=0)
display(ppc_h.round(4))

p = figure(width=860, height=430, title="Hierarchical posterior predictive check by group",
           x_range=ppc_h["group"].tolist(), y_range=(0,1), y_axis_label="survival rate")
p.segment(x0=ppc_h["group"], x1=ppc_h["group"], y0=ppc_h["pred_lower"], y1=ppc_h["pred_upper"], line_width=5)
p.scatter(x=ppc_h["group"], y=ppc_h["pred_mean"], size=9, legend_label="posterior predictive mean")
p.scatter(x=ppc_h["group"], y=ppc_h["observed_rate"], size=10, marker="diamond", legend_label="observed rate")
p.xaxis.major_label_orientation = 0.8
p.legend.location = "top_left"
show(p)

# Part V — Bayesian logistic regression with Bambi
## 13. Prepare passenger-level modelling data

The hierarchical aggregated model only used Sex and Pclass. To model individual survival we add continuous passenger attributes.

We create:

- standardized Age;
- standardized \(\log(1+Fare)\), which reduces the severe right skew of Fare;
- family size = SibSp + Parch + 1;
- categorical Sex, Pclass and Embarked.

Missing Age is imputed with the median **for this teaching GLM**. In a fully Bayesian analysis, missing ages could instead be modelled as latent uncertain quantities.

In [ ]:
model_df = df[["Survived", "Sex", "Pclass", "Age", "Fare", "SibSp", "Parch", "Embarked"]].copy()
model_df["Age"] = model_df["Age"].fillna(model_df["Age"].median())
model_df["Embarked"] = model_df["Embarked"].fillna(model_df["Embarked"].mode().iloc[0])
model_df["logFare"] = np.log1p(model_df["Fare"])
model_df["FamilySize"] = model_df["SibSp"] + model_df["Parch"] + 1
model_df["z_Age"] = (model_df["Age"] - model_df["Age"].mean()) / model_df["Age"].std(ddof=1)
model_df["z_logFare"] = (model_df["logFare"] - model_df["logFare"].mean()) / model_df["logFare"].std(ddof=1)
model_df["Pclass"] = model_df["Pclass"].astype("category")
model_df["Sex"] = model_df["Sex"].astype("category")
model_df["Embarked"] = model_df["Embarked"].astype("category")

display(model_df.head())
display(model_df.describe(include="all").T)

## 14. Bayesian Bernoulli GLM

For passenger \(i\):

\[
Y_i\sim Bernoulli(p_i),
\]

\[
\operatorname{logit}(p_i)=
\beta_0 + X_i^T\beta.
\]

Bambi gives us an R-style formula interface while PyMC remains the inference backend.

The first model is intentionally additive:

\[
Survived \sim Sex + Pclass + zAge + zLogFare + FamilySize + Embarked.
\]

Bambi's automatically scaled priors are useful defaults for teaching, and printing the model makes those priors inspectable rather than hidden.

In [ ]:
formula_base = "Survived ~ Sex + Pclass + z_Age + z_logFare + FamilySize + Embarked"

bambi_base = bmb.Model(formula_base, model_df, family="bernoulli")
print(bambi_base)

idata_base = bambi_base.fit(
    draws=CFG.draws,
    tune=CFG.tune,
    chains=CFG.chains,
    cores=min(CFG.chains, 4),
    random_seed=SEED,
    nuts={"target_accept": CFG.target_accept},
    idata_kwargs={"log_likelihood": True},
)

display(az.summary(idata_base, round_to=3).head(25))
print("Divergences:", divergence_count(idata_base))

### How to interpret logistic coefficients

A coefficient \(\beta_j\) lives on the log-odds scale.

\[
\text{odds ratio}=e^{\beta_j}.
\]

Rather than reporting only a posterior mean, Bayesian analysis can report:

- an interval for \(\beta_j\);
- an interval for \(e^{\beta_j}\);
- \(P(\beta_j>0\mid y)\) or \(P(\beta_j<0\mid y)\).

For categorical predictors, interpretation is always relative to the reference level encoded by the model matrix.

In [ ]:
summary_base = az.summary(idata_base, round_to=4)
# Keep coefficient-like rows; response parameters, if present, can be numerous.
coef_summary = summary_base.copy()
coef_summary["odds_ratio_at_mean"] = np.exp(coef_summary["mean"].clip(-20, 20))
display(coef_summary.head(20))

## 15. A richer interaction model

Titanic survival rules were not additive in a simple sense. The effect of passenger class plausibly differs by sex.

We therefore fit

\[
Survived \sim Sex * Pclass + zAge + zLogFare + FamilySize + Embarked.
\]

The `*` expands to both main effects and the Sex:Pclass interaction.

The purpose is not to assume that a more complex model is automatically better. We will compare predictive performance with PSIS-LOO.

In [ ]:
formula_interaction = "Survived ~ Sex * Pclass + z_Age + z_logFare + FamilySize + Embarked"

bambi_interaction = bmb.Model(formula_interaction, model_df, family="bernoulli")
idata_interaction = bambi_interaction.fit(
    draws=CFG.draws,
    tune=CFG.tune,
    chains=CFG.chains,
    cores=min(CFG.chains, 4),
    random_seed=SEED + 1,
    nuts={"target_accept": CFG.target_accept},
    idata_kwargs={"log_likelihood": True},
)

display(az.summary(idata_interaction, round_to=3).head(30))
print("Divergences:", divergence_count(idata_interaction))

# Part VI — Variational Bayes
## 16. ADVI: turn inference into optimization

MCMC attempts to characterize the target posterior through correlated samples.

Variational inference chooses a tractable family \(q_\phi(\theta)\) and optimizes parameters \(\phi\) so that \(q\) approximates the posterior. Mean-field ADVI typically uses a factorized Gaussian representation in transformed parameter space.

PyMC exposes this directly through:

```python
pm.fit(method="advi")
```

We fit a direct PyMC logistic model so that we can compare **NUTS and ADVI on exactly the same parameterization**.

In [ ]:
# Explicit numeric design matrix for a transparent PyMC comparison.
X = pd.DataFrame({
    "male": (model_df["Sex"].astype(str) == "male").astype(float),
    "class2": (model_df["Pclass"].astype(str) == "2").astype(float),
    "class3": (model_df["Pclass"].astype(str) == "3").astype(float),
    "z_Age": model_df["z_Age"].astype(float),
    "z_logFare": model_df["z_logFare"].astype(float),
    "FamilySize_z": ((model_df["FamilySize"] - model_df["FamilySize"].mean()) / model_df["FamilySize"].std(ddof=1)).astype(float),
})
y = model_df["Survived"].to_numpy(int)
feature_names = X.columns.tolist()
X_np = X.to_numpy(float)

coords_lr = {"feature": feature_names, "obs_id": np.arange(len(y))}

with pm.Model(coords=coords_lr) as logistic_pymc:
    X_data = pm.Data("X_data", X_np, dims=("obs_id", "feature"))
    intercept = pm.Normal("intercept", 0, 1.5)
    beta = pm.Normal("beta", 0, 1.0, dims="feature")
    eta = intercept + pm.math.dot(X_data, beta)
    p = pm.Deterministic("p", pm.math.sigmoid(eta), dims="obs_id")
    obs = pm.Bernoulli("obs", p=p, observed=y, dims="obs_id")

    idata_lr_nuts = pm.sample(
        draws=CFG.draws,
        tune=CFG.tune,
        chains=CFG.chains,
        cores=min(CFG.chains, 4),
        random_seed=SEED,
        nuts={"target_accept": CFG.target_accept},
    )

    advi = pm.fit(
        n=CFG.advi_steps,
        method="advi",
        random_seed=SEED,
        progressbar=True,
    )
    idata_lr_advi = advi.sample(draws=max(1000, CFG.draws * CFG.chains), random_seed=SEED)

print("NUTS diagnostics")
display(diagnostic_table(idata_lr_nuts, ["intercept", "beta"]))

In [ ]:
def coefficient_moments(idata, label):
    b = np.asarray(idata.posterior["beta"]).reshape(-1, len(feature_names))
    rows = []
    for j, name in enumerate(feature_names):
        rows.append({
            "method": label,
            "coefficient": name,
            "mean": b[:, j].mean(),
            "sd": b[:, j].std(ddof=1),
            "q025": np.quantile(b[:, j], 0.025),
            "q975": np.quantile(b[:, j], 0.975),
        })
    return pd.DataFrame(rows)

mom_nuts = coefficient_moments(idata_lr_nuts, "NUTS")
mom_advi = coefficient_moments(idata_lr_advi, "ADVI")
advi_compare = pd.concat([mom_nuts, mom_advi], ignore_index=True)
display(advi_compare.round(4))

In [ ]:
p = figure(width=900, height=440, title="NUTS versus ADVI posterior coefficient uncertainty",
           x_range=feature_names, y_axis_label="coefficient")

for method, offset in [("NUTS", -0.12), ("ADVI", 0.12)]:
    sub = advi_compare[advi_compare["method"] == method]
    # categorical x positions cannot be offset numerically, so use dodge transform through factor tuples
    x = [(name, method) for name in sub["coefficient"]]

# Use a grouped categorical axis for a clean comparison.
p = figure(width=900, height=440, title="NUTS versus ADVI posterior coefficient uncertainty",
           x_range=[(f, m) for f in feature_names for m in ["NUTS", "ADVI"]], y_axis_label="coefficient")
for method in ["NUTS", "ADVI"]:
    sub = advi_compare[advi_compare["method"] == method]
    factors = [(f, method) for f in sub["coefficient"]]
    p.segment(x0=factors, x1=factors, y0=sub["q025"], y1=sub["q975"], line_width=4)
    p.scatter(x=factors, y=sub["mean"], size=9, legend_label=method)
p.xaxis.major_label_orientation = 0.8
p.legend.location = "top_left"
show(p)

### ADVI trade-off

ADVI is usually much faster because inference becomes optimization. But a mean-field approximation cannot represent arbitrary posterior dependence and can underestimate posterior uncertainty.

Therefore the comparison should focus not only on posterior means but also on **standard deviations and interval widths**.

A common professional workflow is:

1. use VI for rapid iteration or very large problems;
2. validate against a stronger MCMC fit on a representative version of the model when feasible.

# Part VII — Model comparison with PSIS-LOO
## 17. Compare additive and interaction GLMs

LOO asks how well a fitted model would predict observations that were left out of fitting. PSIS provides an efficient approximation using posterior draws from the full fit.

The key quantity is expected log pointwise predictive density (ELPD): **larger is better**.

A difference should be interpreted together with its uncertainty and Pareto-\(k\) diagnostics. Model comparison is not a tournament where the numerically top model automatically wins.

In [ ]:
# Compatibility with both the newer split ArviZ packages and older monolithic ArviZ.
try:
    from arviz_stats import loo as loo_fn, compare as compare_fn
except ImportError:
    loo_fn, compare_fn = az.loo, az.compare

loo_base = loo_fn(idata_base, pointwise=True)
loo_interaction = loo_fn(idata_interaction, pointwise=True)

print("Base model LOO")
display(loo_base)
print("Interaction model LOO")
display(loo_interaction)

compare_df = compare_fn({"additive": idata_base, "interaction": idata_interaction})
display(compare_df)

In [ ]:
# Robustly discover the ELPD column across ArviZ versions.
score_col = next((c for c in ["elpd_loo", "elpd"] if c in compare_df.columns), None)
if score_col is not None:
    cmp = compare_df.reset_index(names="model")
    p = figure(width=700, height=340, title="PSIS-LOO predictive comparison",
               x_range=cmp["model"].tolist(), y_axis_label=score_col)
    p.vbar(x=cmp["model"], top=cmp[score_col], width=0.6)
    p.add_tools(HoverTool(tooltips=[("model", "@x"), ("ELPD", "@top{0.000}")]))
    show(p)
else:
    print("Inspect compare_df above; your ArviZ version uses different comparison column names.")

## 18. Pareto-\(k\): which observations make LOO difficult?

PSIS-LOO supplies a Pareto tail-shape diagnostic for each observation. Large \(k\) values indicate that importance sampling is unstable for that observation.

This is particularly useful because it does more than say "LOO has a problem"—it helps identify **which data points deserve inspection**.

In [ ]:
def pareto_table(loo_result, data, top=12):
    if not hasattr(loo_result, "pareto_k") or loo_result.pareto_k is None:
        return pd.DataFrame()
    k = np.asarray(loo_result.pareto_k).reshape(-1)
    out = data[["Survived", "Sex", "Pclass", "Age", "Fare"]].reset_index(drop=True).copy()
    out["pareto_k"] = k[:len(out)]
    return out.sort_values("pareto_k", ascending=False).head(top)

pareto_top = pareto_table(loo_interaction, model_df)
display(pareto_top)

# Part VIII — Posterior prediction and calibration
## 19. Posterior predictive probabilities from Bambi

For a Bernoulli GLM there are two related predictive objects:

1. posterior draws of the **response probability** \(p_i\);
2. posterior predictive draws of a future binary response \(\tilde Y_i\).

The first describes uncertainty in the latent success probability. The second additionally includes Bernoulli outcome randomness.

In [ ]:
# response_params adds posterior draws of the Bernoulli probability parameter.
bambi_interaction.predict(idata_interaction, kind="response_params", inplace=True, random_seed=SEED)
# response adds posterior predictive Bernoulli draws.
bambi_interaction.predict(idata_interaction, kind="response", inplace=True, random_seed=SEED)

print("Posterior variables:", list(idata_interaction.posterior.data_vars))
print("Posterior predictive variables:", list(idata_interaction.posterior_predictive.data_vars))

In [ ]:
# Find the probability parameter generated by the Bernoulli family.
prob_candidates = [v for v in idata_interaction.posterior.data_vars if v in {"p", "mu"}]
if not prob_candidates:
    # In most modern Bambi Bernoulli models it is named 'p'.
    prob_candidates = [v for v in idata_interaction.posterior.data_vars if idata_interaction.posterior[v].ndim >= 3]
prob_var = prob_candidates[0]

prob_draws = np.asarray(idata_interaction.posterior[prob_var])
prob_flat = prob_draws.reshape(-1, prob_draws.shape[-1])
p_mean = prob_flat.mean(axis=0)
p_lo = np.quantile(prob_flat, 0.025, axis=0)
p_hi = np.quantile(prob_flat, 0.975, axis=0)

auc = roc_auc_score(y, p_mean)
brier = brier_score_loss(y, p_mean)
metric_table = pd.DataFrame({
    "metric": ["ROC AUC", "Brier score"],
    "value": [auc, brier],
    "Bayesian interpretation note": [
        "ranking quality using posterior-mean probabilities",
        "proper scoring rule for probabilistic predictions; lower is better"
    ]
})
display(metric_table.round(4))

### Calibration table

Classification accuracy discards most of what a Bayesian probability model produces. Calibration asks whether passengers assigned approximately 70% survival probability survive roughly 70% of the time.

We bin posterior-mean probabilities only for visualization; the posterior itself remains continuous.

In [ ]:
cal = pd.DataFrame({"y": y, "p_mean": p_mean})
cal["bin"] = pd.qcut(cal["p_mean"], q=8, duplicates="drop")
cal_table = cal.groupby("bin", observed=True).agg(
    n=("y", "size"),
    mean_pred=("p_mean", "mean"),
    observed_rate=("y", "mean")
).reset_index()
display(cal_table.round(4))

p = figure(width=650, height=430, title="Calibration: posterior-mean probability versus observed survival",
           x_axis_label="mean predicted probability", y_axis_label="observed rate", x_range=(0,1), y_range=(0,1))
p.line([0,1], [0,1], line_dash="dashed", line_width=2, legend_label="perfect calibration")
p.scatter(cal_table["mean_pred"], cal_table["observed_rate"], size=10)
for _, r in cal_table.iterrows():
    p.text(x=[r["mean_pred"]], y=[r["observed_rate"]], text=[str(int(r["n"]))], x_offset=6, y_offset=4)
p.legend.location = "top_left"
show(p)

## 20. Posterior predictive check for aggregate statistics

A posterior predictive check should target statistics that matter scientifically.

For Titanic we inspect:

- total number of survivors;
- survival rates within Sex × Pclass groups.

If observed statistics repeatedly fall in the tails of the replicated distributions, the model is failing to reproduce an important aspect of the data.

In [ ]:
# The posterior predictive response variable normally has the same name as the response.
pp_var = "Survived" if "Survived" in idata_interaction.posterior_predictive.data_vars else list(idata_interaction.posterior_predictive.data_vars)[0]
pp_y = np.asarray(idata_interaction.posterior_predictive[pp_var])
pp_y_flat = pp_y.reshape(-1, pp_y.shape[-1])

rep_total = pp_y_flat.sum(axis=1)
obs_total = y.sum()

hist, edges = np.histogram(rep_total, bins=35, density=True)
p = figure(width=780, height=370, title="Posterior predictive check: total survivors",
           x_axis_label="replicated total survivors", y_axis_label="density")
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.5)
p.add_layout(Span(location=obs_total, dimension="height", line_dash="dashed", line_width=3))
show(p)

In [ ]:
ppc_rows = []
sex_vals = model_df["Sex"].astype(str).to_numpy()
class_vals = model_df["Pclass"].astype(str).to_numpy()

for sex in sorted(pd.unique(sex_vals)):
    for pc in sorted(pd.unique(class_vals)):
        mask = (sex_vals == sex) & (class_vals == pc)
        if mask.sum() == 0:
            continue
        rep_rates = pp_y_flat[:, mask].mean(axis=1)
        ppc_rows.append({
            "group": f"{sex} / class {pc}",
            "n": int(mask.sum()),
            "observed": y[mask].mean(),
            "pred_mean": rep_rates.mean(),
            "pred_lower": np.quantile(rep_rates, 0.025),
            "pred_upper": np.quantile(rep_rates, 0.975),
        })

ppc_groups = pd.DataFrame(ppc_rows)
display(ppc_groups.round(4))

p = figure(width=860, height=430, title="Posterior predictive check: Sex × Pclass survival rates",
           x_range=ppc_groups["group"].tolist(), y_range=(0,1), y_axis_label="survival rate")
p.segment(x0=ppc_groups["group"], x1=ppc_groups["group"], y0=ppc_groups["pred_lower"], y1=ppc_groups["pred_upper"], line_width=5)
p.scatter(x=ppc_groups["group"], y=ppc_groups["pred_mean"], size=9, legend_label="replicated mean")
p.scatter(x=ppc_groups["group"], y=ppc_groups["observed"], size=10, marker="diamond", legend_label="observed")
p.xaxis.major_label_orientation = 0.8
p.legend.location = "top_left"
show(p)

# Part IX — Synthesis
## 21. What each inferential engine is doing

| Method | Mathematical idea | Strength | Main limitation |
|---|---|---|---|
| Exact conjugacy | recognize an analytic posterior family | exact and extremely fast | available only for special models |
| Metropolis-Hastings | proposal + accept/reject Markov chain | very general | can mix slowly; proposal tuning matters |
| Gibbs sampling | sample from full conditionals | simple when conditionals are standard | dependence can produce slow mixing |
| NUTS | adaptive Hamiltonian trajectories | excellent default for many differentiable continuous models | gradients/geometry matter; discrete parameters need other steps |
| ADVI | optimize a tractable variational approximation | fast and scalable | approximation bias; mean-field may understate dependence/variance |
| PSIS-LOO | importance-sampling approximation to leave-one-out prediction | principled predictive comparison without refitting N times | Pareto-k must be checked |

The central lesson is not "always use NUTS". The lesson is to match the computational method to the mathematical structure of the model.

## 22. Frequentist versus Bayesian quantities

These are related but should not be given interchangeable interpretations.

| Frequentist quantity | Bayesian analogue | Key interpretational difference |
|---|---|---|
| point estimate \(\hat\theta\) | posterior mean/median/MAP | Bayesian estimate is a functional of a posterior distribution |
| confidence interval | credible interval / HDI | credible interval directly represents posterior probability under the model |
| p-value | posterior probability or Bayes factor | p-value is not \(P(H_0\mid y)\) |
| regularization penalty | prior distribution | prior has a probabilistic interpretation |
| bootstrap uncertainty | posterior uncertainty | generated from different inferential constructions |
| train/test score | posterior predictive / LOO score | Bayesian workflow propagates parameter uncertainty into prediction |

## 23. Efficiency and computational complexity

Bayesian model complexity is often dominated by **log-density and gradient evaluation**, repeated across draws and chains.

Let:

- \(n\) = observations;
- \(p\) = predictors/parameters;
- \(S\) = posterior iterations;
- \(C\) = chains.

A logistic-regression log-likelihood evaluation is roughly \(\Theta(np)\). A rough MCMC cost is therefore at least proportional to

\[
\Theta(CSnp),
\]

although NUTS performs multiple gradient evaluations per iteration and its true cost depends strongly on posterior geometry.

### Practical efficiency tips

1. **Standardize continuous predictors.** Posterior geometry becomes easier for NUTS.
2. **Use weakly informative priors.** They regularize implausible regions and often improve sampling.
3. **Inspect divergences before increasing draws.** More draws do not repair bad geometry.
4. **Reparameterize hierarchical models** when funnels appear; non-centered parameterizations are often useful.
5. **Use vectorized PyMC expressions**, not Python loops over observations.
6. **Use FAST_MODE for model development**, then increase chains/draws for final inference.
7. **Compare ESS per second**, not raw samples per second, when evaluating samplers.
8. **Use VI as an approximation consciously**, and validate it against MCMC when possible.
9. **Model comparison needs log likelihoods**; request them during Bambi fitting rather than recomputing repeatedly.
10. **Posterior predictive checking comes before celebrating a good LOO score.** A model can rank best among candidates and still be inadequate.

## 24. Exercises

### Exercise 1 — Prior sensitivity for a rare group
Take the smallest Sex × Pclass group. Fit several Beta priors and quantify how its posterior changes. Explain why the sensitivity differs from the full-data experiment.

### Exercise 2 — Prior predictive checking
For the hierarchical model, change the prior on `kappa`. Before fitting, use `pm.sample_prior_predictive()` to inspect how dispersed the implied group survival probabilities are. Which priors generate absurd worlds?

### Exercise 3 — NUTS versus Metropolis
Increase both samplers to the same number of raw posterior draws. Compare ESS and wall-clock time. Which sampler gives more effective draws per second?

### Exercise 4 — Non-centered hierarchical logistic regression
Replace the aggregated Beta-Binomial model with a passenger-level hierarchical logistic model containing varying intercepts by passenger class or family grouping. Compare centered and non-centered parameterizations.

### Exercise 5 — Missing Age as a latent variable
Instead of median imputation, build a probabilistic model for missing Age and propagate age uncertainty into survival predictions.

### Exercise 6 — Full-rank ADVI
Replace `method="advi"` with `method="fullrank_advi"`. Does the approximation to NUTS improve for correlated coefficients? What happens to runtime?

### Exercise 7 — Decision threshold under asymmetric cost
Suppose a false negative costs five times as much as a false positive. Derive the posterior expected loss of a decision and choose actions using expected utility rather than a fixed 0.5 threshold.

### Exercise 8 — Model expansion and LOO
Add `Age × Sex` or nonlinear age effects. Use PSIS-LOO and Pareto-k diagnostics to determine whether predictive performance actually improves.

## 25. Final mental model

The entire notebook can be compressed into one workflow:

\[
\boxed{
\text{Question}
\rightarrow
\text{probability model}
\rightarrow
\text{prior}
\rightarrow
\text{prior predictive implications}
\rightarrow
\text{data}
\rightarrow
\text{posterior computation}
\rightarrow
\text{diagnostics}
\rightarrow
\text{posterior predictive checks}
\rightarrow
\text{prediction / decision}
}
\]

And the computational choice follows the model:

\[
\boxed{
\text{closed form if available}
\;\rightarrow\;
\text{NUTS/MCMC for accurate general inference}
\;\rightarrow\;
\text{VI when an approximation is justified by scale/speed}
}
\]

The most important ST4234 habit is to stop thinking of an unknown parameter as "one best number." Bayesian statistics treats uncertainty as a first-class mathematical object and carries that uncertainty through every subsequent prediction and decision.

## References and current API notes

### Dataset
- Kaggle Titanic competition schema; notebook downloads an online mirror of `train.csv` and includes an embedded fallback.

### Bayesian software documentation cross-checked for this notebook
- **PyMC `pm.sample`**: https://www.pymc.io/projects/docs/en/stable/api/generated/pymc.sample.html
- **PyMC posterior predictive sampling**: https://www.pymc.io/projects/docs/en/stable/api/generated/pymc.sample_posterior_predictive.html
- **PyMC ADVI / `pm.fit`**: https://www.pymc.io/projects/docs/en/stable/api/generated/pymc.fit.html
- **PyMC multivariate Normal / LKJ**: https://www.pymc.io/projects/docs/en/stable/api/distributions/generated/pymc.MvNormal.html
- **Bambi model API**: https://bambinos.github.io/bambi/api/Model.html
- **Bambi changelog**: https://bambinos.github.io/bambi/changelog.html
- **ArviZ / PSIS-LOO**: https://python.arviz.org/en/stable/api/generated/arviz.loo.html
- **ArviZ model comparison**: https://python.arviz.org/en/stable/api/generated/arviz.compare.html

The notebook intentionally uses package APIs for the substantive inference. Small analytical calculations are retained when they are exact and pedagogically clearer than invoking an MCMC engine.